---
title: "Harvesting Collection (Meta)Data from an API"
---

# Extract (Harvest) Metadata

This section demonstrates the first step in the generalized workflow
described as ETL, or "Extract - Transform - Load."
We will walk through each step in class, but you will need to adapt these demonstrations
so that they work for your own collections, so this work will also be self-guided.

:::{important} Learning Objectives
After completing this section, you should:

- Be able to explain the concept of metadata harvesting and its relationship to data extraction.
- Design and implement a process of harvesting, or extracting, metadata from a known collection.
- Have a conceptual and a practical understanding of how collection metadata is made available by a REST API.
- Understand how the model ETL workflow can be applied to metadata management and digital collections.
% for transformation step:* Use programming to work with data supplied by an API in JSON format, to manage and transform useful parts of that data into CSV format.
% for transformation step: * Create ingest-ready collection metadata that conforms to Dublin Core and other digital collection metadata standards, which can be used to load content into another site (in this case, an Omeka S site). 
:::

## ETL Workflow

The overarching process here follows a generalized "Extract - Transform - Load" workflow.
This is a generalized workflow model for pulling data from one system, transporting and cleaning (meta)data, then loading that product into another system.
While often used to describe moving data between databases or data engineering,
similar overall steps describe the process of collecting and regrouping digital collections.
The process for collections may be described as follows:

Extract
: Pull the metadata you will need from source system(s); in the case of the workflow outlined here, this step harvests metadata about a known list of items from the Library of Congress using the Library's JSON API. 

Transform
: Change the data using string manipulations, or other data management, into structures that will make sense to the target collection system(s); in the case of the workflow outlined in this book, that includes both CollectionBuilder and Omeka S.

Load
: Finally, then ingest the transformed (meta)data and associated content into the target system(s); in the workflow outlined here, the primary data structure is CSV, although these systems can import data in multiple formats.

## Metadata Harvesting

This section outlines the *extract* step through a process that may be described in digital collections management as **metadata harvesting**.
The details of this step will vary depending on the source system, so this process can take shape in many diverse ways. As illustrated here, the process operates by identifying a known collection (i.e., a set of items), then requests the associated metadata and content files using a REST API call.
Specifically, the process makes use of the Library of Congress' JSON API.

There are three largescale steps in this harvesting sequence:

1. Get collections list - using the requests library, make a request to the library of congress API to get the list of items in the "Free to Use" libraries collection. Write this to a local file (here called `collection_items_list.csv` and in the `data` directory). 
2. Get item metadata - using the collections list from step one as a feeder file, request JSON data for each item in the list.
3. Get item content - using data from each item's metadata, download at least one file for each of the items in the list. Although this is an important step in the harvesting process, this section will only focus on harvesting descriptive metadata and leave the content step for later.

# 1. Get collection list

:::{attention} Dependencies for this Section
This section uses the following Python modules:

- Requests
- JSON
- CSV
:::

To get set up, import the libraries required for this section:

In [12]:
import csv
import json
import requests

## Build the URL

Using the requests library, use the patterns you developing in [](/labs/lab-05-python-urls-requests-json.md),
to construct a URL. Using the conventions discussed in the previous section, this demonstration uses a base URL and endpoint pattern.

In [2]:
baseURL = 'https://www.loc.gov'
endpoint = '/free-to-use'
parameters = {
    'fo' : 'json'
}

Since there are many collection sets in the Library's "Free to Use" items, this URL will also require an additional variable, which can be named collection: 

In [3]:
collection = '/libraries'

Then, assemble the pieces into a URL and use the requests library send a get request to retrieve the JSON representing the collection list.

In [4]:
collection_list_response = requests.get(baseURL + endpoint + collection, params=parameters)

print(collection_list_response.url)

https://www.loc.gov/free-to-use/libraries?fo=json


In [5]:
collection_list_response.status_code

200

A status code value of `200` indicates that the response has been successful! 🎉

## Examine the response

Look at the JSON response and find the data you want: the collection set list.
The code below will query LOC for the collection list and print the response; due to length, the output is hidden.
You should notice that the response is quite lengthy,[^how-long] but look for a section that appears to have information about the items in this free to use set. Hint: the information is minimal, and it is all in one array, which indicates the "items," and shows a URL, label, and a thumbnail image.

[^how-long]: As of October 2026, the JSON response for the collection list is more than 3,000 lines.

In [6]:
collection_json = collection_list_response.json()

print(json.dumps(collection_json, indent=2))

{
  "breadcrumbs": [
    {
      "Library of Congress": "https://www.loc.gov"
    },
    {
      "Library of Congress Free to Use and Reuse": "https://www.loc.gov/free-to-use/"
    },
    {
      "Libraries": "https://www.loc.gov/free-to-use/libraries/"
    }
  ],
  "content": {
    "active": true,
    "components": [
      {
        "blurb": "For greater knowledge, on more subjects, use your library more often. Illinois WPA Arts Project, 1936-1941. Prints & Photographs Division, Noyes Library for Young Children. Kensington, Maryland. Photo by Carol M. Highsmith,  2011. Prints & Photographs Division, Bethune-Cookman College. Students in the library reading room, Daytona Beach, Florida. Gordon Parks, 1943. Prints & Photographs Division, Public library in Antonito,  Colorado, near the New Mexico border. Photo by Carol M. Highsmith, 2015. Prints & Photographs Division, Library and Innovation Center in New Castle, Delaware, defines itself as \"a community learning and resource hub.\" Photo

### Examining the Response with Python

It's entirely possible to read through the response and identify the elements. Aside from that, however, there are a python tools that can also help to examine the response.

In [13]:
# .keys() is a helpful function to see what the data elements are
collection_json.keys()

dict_keys(['breadcrumbs', 'content', 'description', 'disable_max_line_length', 'disable_navigation', 'expert_resources', 'manifest', 'next', 'next_sibling', 'options', 'pages', 'portal', 'portal_label', 'previous', 'previous_sibling', 'site_type', 'timestamp', 'title', 'type', 'ui2'])

Looking further into the dictionary, it seems that you can get a list of the items in the set by looking into `content`, then `components`, then `masonry_gallery`, and finally in the `items` element. Note that a few of these, specifically the components and gallery are arrays, it is necessary to reference the first element in the array using the zero index. 

In [9]:
for k in collection_json['content']['components'][0]['masonry_gallery'][0]['items']:
    print(k)

{'image': '/static/portals/free-to-use/public-domain/libraries/libraries-1.jpg', 'link': '/resource/cph.3f05183/', 'title': 'For greater knowledge, on more subjects, use your library more often. Illinois WPA Arts Project, 1936-1941. Prints & Photographs Division'}
{'image': '/static/portals/free-to-use/public-domain/libraries/libraries-2.jpg', 'link': '/resource/highsm.20336/', 'title': 'Noyes Library for Young Children. Kensington, Maryland. Photo by Carol M. Highsmith,  2011. Prints & Photographs Division'}
{'image': '/static/portals/free-to-use/public-domain/libraries/libraries-3.jpg', 'link': '/resource/fsa.8d24709/', 'title': 'Bethune-Cookman College. Students in the library reading room, Daytona Beach, Florida. Gordon Parks, 1943. Prints & Photographs Division'}
{'image': '/static/portals/free-to-use/public-domain/libraries/libraries-4.jpg', 'link': '/resource/highsm.36052/', 'title': 'Public library in Antonito,  Colorado, near the New Mexico border. Photo by Carol M. Highsmith,

A copy of the JSON output retrieved above can be found in the data files at [](/data/apis-02-collection-list.json). The first few items in the section can be viewed below. Note in particular the `link` elements, which will be used next to formulate URLs for each individual item.

```{literalinclude} /data/apis-02-collection-list.json
:start-at: items
:end-before: libraries-4
:lineno-match: 
:emphasize-lines: 28, 33, 38
```

To double check the work later, you should check how many items are in the set, which can be done by tallying the "length" of the list:

In [10]:
len(collection_json['content']['components'][0]['masonry_gallery'][0]['items'])

62

## Save the List to a CSV

In a more fully automated environment, you might want to make a function that can return and save the collection list, then reuse it in other code, but for the purposes of demonstrating the workflow, it is useful to save the information. So to complete this step, extract the item links in the list and save them locally to a CSV. 

In [14]:
# create a path for a CSV file, in this case to write to the collection-project directory
collection_set_list = '../output/collection_set_list.csv'
headers = ['image','link','title']

with open(collection_set_list, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    for item in collection_json['content']['components'][0]['masonry_gallery'][0]['items']:
        
        # clean up errant spaces in the title fields
        item['title'] = item['title'].rstrip()
        writer.writerow(item)
    print('wrote',collection_set_list)

wrote ../output/collection_set_list.csv


Saving the list to a CSV creates a re-usable collection list to work from in the future.
This will be useful to feed your collection workflows in future, when you want to
perform batch operations for each thing in the collection.

The next step in this workflow development assignment is to harvest metadata for each of the individual items.